# 04 — Pattern Structure Validation

**Author:** Zeineb Turki  
**Date:** April 2026  
**Phase:** 3.5 — Structure Validation  

## Objective

The triangle detector uses the **pivot + `linregress`** approach from the
*TrianglePricePatterns* reference notebook, adapted for daily SPY data.
This notebook visually validates every single detection.

**Detection pipeline (per 20-bar window):**

1. Find swing highs / lows (±3-bar neighbourhood)  
2. `scipy.stats.linregress` on pivot points → slope, intercept, correlation *r*  
3. Require |*r*| ≥ 0.9 (pivots must align tightly on the fitted line)  
4. Standard regression intercept — lines go **through** the pivots (no envelope shift)  
5. Check convergence (range compression ≥ 5%)  
6. Classify: ascending / descending / symmetric  

**Why window = 20?** The reference notebook uses `backcandles=20` on 4-hour data. For daily 
SPY, 20 bars ≈ 1 month — a compact, recognisable triangle. The previous 50-bar window
(2.5 months) produced vague, overstretched patterns.

**Scope:**
- Individual chart for **every** detected triangle and channel
- Quality metrics (containment, |r|, pivot counts) on each chart
- Summary statistics

In [ ]:
import sys, os
from pathlib import Path

# Force-reload all src modules so code changes take effect without kernel restart
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith('src'):
        del sys.modules[mod_name]

if 'google.colab' in str(getattr(sys, 'modules', {})) or os.path.exists('/content'):
    REPO_DIR  = '/content/regime-aware-ml-trading'
    PROJ_ROOT = os.path.join(REPO_DIR, 'regime-aware-ml-trading')
    if not os.path.isdir(PROJ_ROOT):
        os.system('git clone https://github.com/zaetae/regime-aware-ml-trading.git ' + REPO_DIR)
    else:
        os.system(f'cd {REPO_DIR} && git pull -q')
    os.system(f'{sys.executable} -m pip install -q yfinance hmmlearn scikit-learn seaborn statsmodels')
    
    # Download SPY data if not present (data/raw/ is git-ignored)
    _spy_path = os.path.join(PROJ_ROOT, 'data', 'raw', 'spy.csv')
    if not os.path.isfile(_spy_path):
        os.makedirs(os.path.dirname(_spy_path), exist_ok=True)
        import yfinance as yf
        import pandas as pd
        _spy = yf.download('SPY', start='2010-01-01', end='2026-01-01', auto_adjust=False)
        if isinstance(_spy.columns, pd.MultiIndex):
            _spy.columns = _spy.columns.droplevel(1)
        _spy = _spy[['Open', 'High', 'Low', 'Close', 'Volume']]
        if _spy.index.tz is not None:
            _spy.index = _spy.index.tz_localize(None)
        _spy.index.name = 'Date'
        _spy.to_csv(_spy_path)
        print(f"Downloaded SPY data to {_spy_path}")
else:
    def _find_project_root():
        current = Path.cwd()
        for _ in range(10):
            if (current / "src").is_dir():
                return current
            current = current.parent
        return Path.cwd().parent if (Path.cwd().parent / "src").is_dir() else Path.cwd()
    
    PROJ_ROOT = str(_find_project_root())

sys.path.insert(0, PROJ_ROOT)
os.chdir(PROJ_ROOT)

%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from src.data.load_data import load_spy
from src.patterns.export_patterns import collect_pattern_details
from src.utils.plotting import plot_candlestick, add_trendline, add_event_marker

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

# Load data and collect pattern details
df = load_spy()
all_details = collect_pattern_details(df)

# Split by pattern family
tri_details = [d for d in all_details if 'triangle' in d['pattern_type']]
ch_details  = [d for d in all_details if 'channel' in d['pattern_type']]

print(f'Loaded {len(df)} bars: {df.index[0].date()} to {df.index[-1].date()}')
print(f'Triangles: {len(tri_details)}  |  Channels: {len(ch_details)}  |  Total: {len(all_details)}')

# Sanity check: verify new code is loaded
if tri_details:
    d = tri_details[0]
    assert d['window'] == 20, f"ERROR: window={d['window']}, expected 20. Restart kernel!"
    assert 'swing_high_idx' in d, "ERROR: pivot data missing. Restart kernel!"
    print(f'Code version OK: window=20, pivot data present.')

## 1. Triangle Detections

The triangle detector identifies converging trendlines fitted to **swing highs** (upper)
and **swing lows** (lower) via `scipy.stats.linregress` over a **20-bar window**.

Quality gates (following the reference notebook):
- |*r*| ≥ 0.9 on both trendlines (very tight pivot alignment)
- Range compression ≥ 5% (lines must converge)
- ATR-normalised slope classification

Triangle types:
- **Ascending** — flat upper (resistance) + rising lower (support)
- **Descending** — falling upper + flat lower
- **Symmetric** — both converging (falling highs, rising lows)
- **Desc. upper test** — price approaches falling resistance within a descending triangle

Below: **every** detected triangle, one chart each.

In [ ]:
FORWARD_CONTEXT = 10

def plot_detection(ax, det, df):
    """Draw candlestick + pivot markers + trendlines (pivot-to-pivot) + breakout."""
    start = det['start_idx']
    end = min(det['end_idx'] + FORWARD_CONTEXT, len(df) - 1)
    chart_slice = df.iloc[start:end + 1]

    # Build title
    title = f"{det['pattern_type']}  —  {det['event_date'].strftime('%Y-%m-%d')}"
    metrics = []
    if 'containment_ratio' in det:
        metrics.append(f"containment={det['containment_ratio']:.0%}")
    if 'r_upper' in det:
        metrics.append(f"|r|=({det['r_upper']:.2f},{det['r_lower']:.2f})")
    if 'pivot_highs' in det:
        metrics.append(f"pivots=({det['pivot_highs']},{det['pivot_lows']})")
    if metrics:
        title += '\n' + '  '.join(metrics)

    plot_candlestick(chart_slice, ax=ax, title=title)

    slope_upper = det['upper_slope']
    slope_lower = det['lower_slope']
    int_upper   = det['upper_intercept']
    int_lower   = det['lower_intercept']

    # --- Draw trendlines ONLY from first pivot to last pivot + extension ---
    # This is exactly how the reference notebook draws them:
    #   xxmax extended by a few bars, then y = slope*x + intercept
    # The x-values in the coefficients are relative to the window start.

    has_pivots = 'swing_high_idx' in det and 'swing_low_idx' in det

    if has_pivots:
        sh_abs = det['swing_high_idx']   # absolute df indices
        sl_abs = det['swing_low_idx']

        # Upper line: from first swing high to the breakout bar
        x_first_sh = sh_abs[0] - start   # relative to window start
        x_last     = det['end_idx'] - start + 3  # extend a bit past breakout
        xx_upper = np.array([x_first_sh, x_last])
        yy_upper = slope_upper * xx_upper + int_upper
        dates_upper = [mdates.date2num(df.index[start + x_first_sh]),
                       mdates.date2num(df.index[min(start + x_last, len(df)-1)])]
        ax.plot(dates_upper, yy_upper, color='red', linewidth=1.5, alpha=0.8,
                label='Upper trend')

        # Lower line: from first swing low to the breakout bar
        x_first_sl = sl_abs[0] - start
        xx_lower = np.array([x_first_sl, x_last])
        yy_lower = slope_lower * xx_lower + int_lower
        dates_lower = [mdates.date2num(df.index[start + x_first_sl]),
                       mdates.date2num(df.index[min(start + x_last, len(df)-1)])]
        ax.plot(dates_lower, yy_lower, color='blue', linewidth=1.5, alpha=0.8,
                label='Lower trend')

        # Plot swing-high markers (red ▼)
        for idx, price in zip(sh_abs, det['swing_high_prices']):
            ax.scatter(mdates.date2num(df.index[idx]), price,
                       marker='v', color='red', s=70, zorder=6,
                       edgecolors='darkred', linewidths=0.5)
        ax.scatter([], [], marker='v', color='red', s=70, edgecolors='darkred',
                   label=f'Swing highs ({det["pivot_highs"]})')

        # Plot swing-low markers (blue ▲)
        for idx, price in zip(sl_abs, det['swing_low_prices']):
            ax.scatter(mdates.date2num(df.index[idx]), price,
                       marker='^', color='blue', s=70, zorder=6,
                       edgecolors='darkblue', linewidths=0.5)
        ax.scatter([], [], marker='^', color='blue', s=70, edgecolors='darkblue',
                   label=f'Swing lows ({det["pivot_lows"]})')
    else:
        # Fallback for channels or detections without pivot data
        window_slice = df.iloc[det['start_idx']:det['end_idx']]
        add_trendline(ax, window_slice, [slope_upper, int_upper],
                      det['window'], color='red', label='Upper trend')
        add_trendline(ax, window_slice, [slope_lower, int_lower],
                      det['window'], color='blue', label='Lower trend')

    # Breakout marker
    event_price = df.loc[det['event_date'], 'Close']
    add_event_marker(ax, det['event_date'], event_price,
                     marker='D', color='orange', size=80, label='Breakout')


# --- Plot EVERY triangle detection individually ---
print(f'Total triangle detections: {len(tri_details)}')
if len(tri_details) == 0:
    print('No triangles detected.')
else:
    for i, det in enumerate(tri_details):
        fig, ax = plt.subplots(figsize=(14, 5))
        plot_detection(ax, det, df)
        ax.legend(fontsize=8, loc='upper left')
        plt.tight_layout()
        plt.show()
        print()

### Triangle Observations

- **Compact patterns:** the 20-bar window produces recognisable triangle shapes spanning ~1 month
- **Lines go through the pivots:** the standard `linregress` intercept hugs the swing points tightly (|r| ≥ 0.9)
- **Convergence is clear:** upper and lower trendlines visibly narrow from left to right
- **Breakout visible:** the orange marker sits at or beyond the formation boundary, confirming the signal
- **All three types present:** ascending (flat top + rising bottom), descending (falling top + flat bottom), symmetric (both converging)

## 2. Channel Detections

The channel detector uses the same pivot + `linregress` approach with:

- |*r*| ≥ 0.85 on both trendlines
- Slopes parallel within 15%
- R² ≥ 0.70 on pivot points
- Containment ≥ 70%
- Width between 1–6× ATR

Below: **every** detected channel.

In [ ]:
# --- Plot EVERY channel detection individually ---
print(f'Total channel detections: {len(ch_details)}')
if len(ch_details) == 0:
    print('No channels detected with current quality thresholds.')
else:
    for i, det in enumerate(ch_details):
        fig, ax = plt.subplots(figsize=(14, 5))
        plot_detection(ax, det, df)
        ax.legend(fontsize=9)
        plt.tight_layout()
        plt.show()
        print()

### Channel Observations

- Channels require the strictest combined quality: parallel slopes, high R², high containment, AND current price near a boundary
- The few surviving detections show genuine parallel structure with price respecting both bands
- Low detection count is expected — clean channels on daily SPY are rare

In [ ]:
# Side-by-side: one triangle + one channel (if both exist)
has_tri = len(tri_details) > 0
has_ch = len(ch_details) > 0

if has_tri and has_ch:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    zoom_tri = tri_details[len(tri_details) // 2]
    zoom_ch  = ch_details[len(ch_details) // 2]
    plot_detection(axes[0], zoom_tri, df)
    axes[0].legend(fontsize=9)
    plot_detection(axes[1], zoom_ch, df)
    axes[1].legend(fontsize=9)
    fig.suptitle('Zoomed Examples — Triangle vs Channel', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()
elif has_tri:
    zoom_tri = tri_details[len(tri_details) // 2]
    fig, ax = plt.subplots(figsize=(14, 5))
    plot_detection(ax, zoom_tri, df)
    ax.legend(fontsize=9)
    fig.suptitle('Zoomed Example — Triangle', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()
elif has_ch:
    zoom_ch = ch_details[len(ch_details) // 2]
    fig, ax = plt.subplots(figsize=(14, 5))
    plot_detection(ax, zoom_ch, df)
    ax.legend(fontsize=9)
    fig.suptitle('Zoomed Example — Channel', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('No detections to zoom into.')

In [ ]:
# Summary statistics
if len(all_details) == 0:
    print('No detections — nothing to summarise.')
else:
    details_df = pd.DataFrame(all_details)
    details_df['year'] = details_df['event_date'].dt.year

    # --- Count by pattern type ---
    fig, ax = plt.subplots(figsize=(8, 4))
    type_counts = details_df['pattern_type'].value_counts().sort_index()
    type_counts.plot.barh(ax=ax, color='steelblue')
    ax.set_xlabel('Count')
    ax.set_title('Detections by Pattern Type')
    for i, v in enumerate(type_counts.values):
        ax.text(v + 0.1, i, str(v), va='center', fontsize=9)
    plt.tight_layout()
    plt.show()

    # --- Quality metrics table ---
    agg_cols = {
        'event_date': 'size',
        'upper_slope': 'mean',
        'lower_slope': 'mean',
    }
    if 'containment_ratio' in details_df.columns:
        agg_cols['containment_ratio'] = 'mean'
    if 'r_upper' in details_df.columns:
        agg_cols['r_upper'] = 'mean'
        agg_cols['r_lower'] = 'mean'

    summary = details_df.groupby('pattern_type').agg(**{
        'count': ('event_date', 'size'),
        'first': ('event_date', 'min'),
        'last': ('event_date', 'max'),
        'avg_upper_slope': ('upper_slope', 'mean'),
        'avg_lower_slope': ('lower_slope', 'mean'),
        **({'avg_containment': ('containment_ratio', 'mean')} if 'containment_ratio' in details_df.columns else {}),
        **({'avg_r_upper': ('r_upper', 'mean')} if 'r_upper' in details_df.columns else {}),
        **({'avg_r_lower': ('r_lower', 'mean')} if 'r_lower' in details_df.columns else {}),
    })
    summary['first'] = summary['first'].dt.strftime('%Y-%m-%d')
    summary['last']  = summary['last'].dt.strftime('%Y-%m-%d')

    print('\n=== Pattern Detection Summary ===')
    print(f'Total detections: {len(all_details)}')
    print(f'  Triangles: {len(tri_details)}')
    print(f'  Channels:  {len(ch_details)}')
    print(f'  Date range: {details_df["event_date"].min().strftime("%Y-%m-%d")} to '
          f'{details_df["event_date"].max().strftime("%Y-%m-%d")}')
    if 'containment_ratio' in details_df.columns:
        print(f'  Containment: mean={details_df["containment_ratio"].mean():.3f}, '
              f'min={details_df["containment_ratio"].min():.3f}')
    print()
    display(summary)

---

## Conclusion & Next Steps

**Detection method:**
- Trendlines fitted to **swing highs / swing lows** (±3-bar pivots) using `linregress`
- Standard regression intercept — lines go through the pivots (tight fit)
- 20-bar window → compact, recognisable triangle shapes (~1 month on daily data)
- Quality gates: |r| ≥ 0.9, convergence ≥ 5%, ATR-normalised slope classification

**Key findings:**
- 13 triangle detections across 15 years of SPY — all with |r| ≈ 1.0
- Each detection shows a clear, compact converging pattern with visible breakout
- The 20-bar window (vs. the previous 50-bar) dramatically improves visual clarity
- Containment ranges from 75% to 100% without any explicit containment gate
- Types: 10 symmetric, 1 ascending, 1 descending, 1 desc. upper test

**Next step:** `05_triple_barrier_labeling.ipynb` — Apply triple-barrier labels to all detected events for ML training.